# Input

In [ ]:
dataName = "IMDBC5"
# dataName = "IMDBLargeC5"
# dataName = "Brazilnew"

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from sklearn.metrics import confusion_matrix
import os
import lightgbm as lgb
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import torch

In [ ]:
X_train = []
X_test = []
y_train = []
y_test = []

In [ ]:
def LoadDataset(dataset):
    global X_train, X_test, X_val, y_val,y_train, y_test
    DATASET_DIR = '/home/jiayi/disk/C-craig/dataset/'
    
    X_train = np.load(DATASET_DIR + "{}-train-X.npy".format(dataset))
    X_val = np.load(DATASET_DIR + "{}-val-X.npy".format(dataset))
    X_test = np.load(DATASET_DIR + "{}-test-X.npy".format(dataset))

    y_train = np.load(DATASET_DIR + "{}-train-y.npy".format(dataset))
    y_val = np.load(DATASET_DIR + "{}-val-y.npy".format(dataset))
    y_test = np.load(DATASET_DIR + "{}-test-y.npy".format(dataset))

In [ ]:
# LoadDataset("IMDBC5")
LoadDataset(dataName)

# Decision Tree

## Full

In [ ]:
from sklearn import tree

clf = tree.DecisionTreeClassifier(min_samples_split=100)

In [ ]:
%%time
clf = clf.fit(X_train, y_train)

In [ ]:
%%time
predictions = clf.predict(X_test)


In [ ]:
%%time
test_predict_y = predictions
print(np.unique(test_predict_y))

full_test_f1 = metrics.f1_score(y_test, test_predict_y , average = 'micro')
full_test_f1_weighted = metrics.f1_score(y_test, test_predict_y , average = 'weighted')
full_test_recall = metrics.recall_score(y_test, test_predict_y , average = 'weighted')
full_test_recall_weighted = metrics.recall_score(y_test, test_predict_y , average = 'weighted')

full_test_accuracy = metrics.accuracy_score(y_test, test_predict_y)
# full_test_f1 = metrics.f1_score(y_test, test_predict_y , average = 'macro')

print("Trained on   ",X_train.shape)
print('F1-score Mircro   [{}]'.format(full_test_f1))
print('F1-score Weighted [{}]'.format(full_test_f1_weighted))
print('Recall            [{}]'.format(full_test_recall))
print("Accuracy          [{}]".format(full_test_accuracy))
# print(confusion_matrix(y_test, test_predict_y))

In [ ]:
full_acc = full_test_accuracy

## Evaluate Func

In [ ]:
def evaluateClass(predictions, y_test):
    test_predict_y = predictions
    print(np.unique(test_predict_y))

    full_test_f1 = metrics.f1_score(y_test, test_predict_y , average = 'micro')
    full_test_f1_weighted = metrics.f1_score(y_test, test_predict_y , average = 'weighted')
    full_test_recall = metrics.recall_score(y_test, test_predict_y , average = 'weighted')
    full_test_recall_weighted = metrics.recall_score(y_test, test_predict_y , average = 'weighted')

    full_test_accuracy = metrics.accuracy_score(y_test, test_predict_y)
    # full_test_f1 = metrics.f1_score(y_test, test_predict_y , average = 'macro')

#     print("Trained on   ",X_train.shape)
    print('F1-score Mircro   [{}]'.format(full_test_f1))
    print('F1-score Weighted [{}]'.format(full_test_f1_weighted))
    print('Recall            [{}]'.format(full_test_recall))
    print("Accuracy          [{}]".format(full_test_accuracy))
    # print(confusion_matrix(y_test, test_predict_y))
    
    return full_test_accuracy

In [ ]:
def testClass(X_train, y_train, X_test, y_test):
    print("Trained on   ", X_train.shape)
    
    clf = tree.DecisionTreeClassifier(splitter='random')
    clf = clf.fit(X_train, y_train)
    predictions = clf.predict(X_test)
    
    acc = evaluateClass(predictions, y_test)
    return acc

## coreset

### Load

In [ ]:
def LoadCoreset(coreset_from, data, subset_size, batch=0,sampleSize=0):
    if coreset_from == 'scratch':
        file_name = ''

    elif coreset_from == 'diskCRAIG':
        file_name = '/home/jiayi/disk/C-craig/inuse/{}-{}.npz'.format(data, subset_size)
    elif coreset_from == 'diskLTLG':
        file_name = '/home/jiayi/disk/C-craig/inuse/{}-{}-ltlg.npz'.format(data, subset_size)
    elif coreset_from == 'diskOurs':
        if batch==0:
            if subset_size == 0.00001:
                file_name = '/home/jiayi/disk/C-craig/inuse/{}-0.00001-ours.npz'.format(data)
            else:
                file_name = '/home/jiayi/disk/C-craig/inuse/{}-{}-ours.npz'.format(data, str(subset_size))
        elif batch==1: 
            file_name = '/home/jiayi/disk/C-craig/BatchGen/{}-{}-coreset.npz'.format(data, str(subset_size))
        elif batch==2: 
            assert sampleSize !=0
            file_name = '/home/jiayi/disk/C-craig/SampleSize/{}-{}-{}-coreset.npz'.format(data, str(subset_size),sampleSize )
        else:
            assert False
    print("【Load file path】 is ", file_name)


    if file_name != '':
        print(f'reading from {file_name}')
        dataset = np.load(f'{file_name}')
        order, weights, total_ordering_time = dataset['order'], dataset['weight'], dataset['order_time']
        print(" 【Coreset size】 is ", order.shape)
        return order, weights, total_ordering_time

In [ ]:
SAMPLE_PROP_DICT = {
    "IMDBC5":0.0128,
    "IMDBLargeC5":0.0016,
    "Brazilnew":0.0016,
    "IMDBCLinear": 0.0032,
    "IMDBLargeCLinear": 0.0016,
    "stackn":0.0032,
    "taxi":0.0032
}

In [ ]:
PROP = SAMPLE_PROP_DICT[dataName]

### Load coreset

In [ ]:

order, weights, _ = LoadCoreset('diskOurs', dataName, subset_size=PROP)

In [ ]:
CS_X_train = X_train[order,:]
CS_W_train = weights.astype(np.int64)
CS_y_train = y_train[order]

In [ ]:
print(CS_W_train)
print(CS_X_train)
print(CS_W_train.shape)
print(CS_X_train.shape)

In [ ]:
new_X_train = np.repeat(CS_X_train, CS_W_train, axis=0)
new_y_train = np.repeat(CS_y_train, CS_W_train, axis=0)

In [ ]:
print(CS_X_train.shape)
print(CS_y_train.shape)

In [ ]:
print(CS_W_train.shape)

In [ ]:
print(new_X_train.shape)
print(new_y_train.shape)

In [ ]:
print(new_X_train.shape)

In [ ]:
ours_acc = testClass(new_X_train, new_y_train, X_test, y_test)

## sample

### Load sample

In [ ]:
# np.shuffle()
idxs = np.arange(X_train.shape[0])
np.random.shuffle(idxs)

idxs = idxs[:(int)(PROP * X_train.shape[0])]
sample_X_train = X_train[idxs,:]
sample_y_train = y_train[idxs].reshape(-1,1)

y_test = y_test.reshape(-1,1)

In [ ]:
%%time
sample_acc = testClass(sample_X_train, sample_y_train, X_test, y_test)

## Output

In [ ]:
print("Full   Acc: ", full_acc)
print("RECON  Acc: ", ours_acc)
print("Sample Acc: ", sample_acc)